# QueryMind - Hello World LLM Call

Minimal end-to-end test of text-to-SQL pipeline.

Entire schema is hardcoded into the prompt - no RAG. No safety validaion.

Simply looking for proof of concept of the basic loop:

    Plain Language question --> LLM --> SQL --> Execute --> Results

In [1]:
!pip install anthropic

## Setup

In [2]:
import sys
from pathlib import Path

project_root = str(Path.cwd().parent)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import anthropic
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import text

from src.database.connection import get_engine

# Load API key from .env file
load_dotenv(Path(project_root) / ".env")

client = anthropic.Anthropic() # Reads ANTHROPIC_API_KEY from env file
engine = get_engine(readonly=True)

print("Anthropic client and database ready.")

Anthropic client and database ready.


## Hardcoded Schmea Context

In [ ]:
# At a later stage, this will be replace by RAG retriveal from ChromaDB
# For our test now, below is a hardcoded, simplified schema description

SCHEMA_CONTEXT = """
DATABASE SCHEMA (SQLite):

Table: olist_orders (99,441 rows) - Central fact table for all orders
    - order_id (TEXT, PK) - Unique order identifier.
    - customer_id (TEXT, FK -> olist_customers.customer_id)
    - order_status (TEXT) — Values: delivered, shipped, canceled, unavailable, invoiced, processing, created, approved. 97% are 'delivered'.
    - order_purchase_timestamp (TEXT) — ISO format datetime, e.g. '2017-10-02 10:56:33'. Data range: Jan 2017 - Aug 2018.
    - order_approved_at (TEXT) - Datetime when payment was approved. NULL for ~0.2% of orders.
    - order_delivered_carrier_date (TEXT) - Datetime when order was handed to carrier. NULL for ~1.8%.
    - order_delivered_customer_date (TEXT) - Datetime of actual delivery. NULL for ~3% (non-delivered orders).
    - order_estimated_delivery_date (TEXT) - Estimated delivery datetime.

Table: olist_order_items (112,650 rows) - Line items within each order
    - order_id (TEXT, FK -> olist_orders.order_id)
    - order_item_id (INTEGER) - Sequence number within the order (1, 2, 3...), NOT a unique ID.
    - product_id (TEXT, FK -> olist_products.product_id)
    - seller_id (TEXT, FK -> olist_sellers.seller_id)
    - shipping_limit_date (TEXT) - Seller's Shipping deadline.
    - price (REAL) - Item price in BRL (Brazilian Real).
    - freight_value (REAL) - Shipping cost for specified item in BRL.

Table: olist_products (32,951 rows) - Product catalog
    - product_id (TEXT, PK) - Unique product identifier.
    - product_category_name (TEXT) - Category in Portuguese. NULL for 1.9% of products.
    - product_weight_g (REAL), product_length_cm (REAL), product_height_cm (REAL), product_width_cm (REAL)

Table: product_category_name_translation (71 rows) - Portuguese to English category mapping
    - product_category_name (TEXT, PK) - Portuguese name (joins to olist_products)
    - product_category_name_english (TEXT) - English translation.

Table: olist_customers (99,441 rows) - Customer information
    - customer_id (TEXT, PK) — Per-order customer ID (FK target for olist_orders)
    - customer_unique_id (TEXT) — True unique customer identifier. Use this for counting distinct customers.
    - customer_zip_code_prefix (INTEGER)
    - customer_city (TEXT), customer_state (TEXT) — 27 Brazilian states

Table: olist_order_payments (103,886 rows) - Payment details per order
    - order_id (TEXT, FK → olist_orders.order_id)
    - payment_sequential (INTEGER) — Payment sequence within order (most orders have 1)
    - payment_type (TEXT) — Values: credit_card (74%), Boleto (19%), voucher (6%), debit_card (1.5%)
    - payment_installments (INTEGER) — Number of installments (1–24)
    - payment_value (REAL) — Amount paid in BRL

Table: olist_order_reviews (99,224 rows) - Customer reviews
    - review_id (TEXT, PK)
    - order_id (TEXT, FK → olist_orders.order_id)
    - review_score (INTEGER) — 1 to 5. Skewed positive: 58% are 5-star.
    - review_comment_title (TEXT) — NULL for 88%
    - review_comment_message (TEXT) — NULL for 59%

Table: olist_sellers (3,095 rows) - Seller information
    - seller_id (TEXT, PK)
    - seller_city (TEXT), seller_state (TEXT)

Table: olist_geolocation (1,000,163 rows) - Zip code to coordinates (one-to-many)
    - geolocation_zip_code_prefix (INTEGER)
    - geolocation_lat (REAL), geolocation_lng(REAL)
    - geolocation_city (REAL), geolocation_state (REAL)

KEY RELATIONSHIPS:
    olist_orders.customer_id -> olist_customers.customer_id
    olist_order_items.order_id -> olist_orders.order_id
    olist_order_items.product_id -> olist_products.product_id
    olist_order_items.seller_id -> olist_sellers.seller_id
    olist_order_payments.order_id -> olist.orders.order_id
    olist_order_reviews.order_id -> olist_orders.order_id
    olist_products.product_category_name -> product_category_name_translation.product_category_name (use LEFT JOIN - 2 categories lack translations)

BUSINESS RULES:
    - Revenue = SUM(olist_order_items.price + olist_order_items.freight_value).
    - Delivery time = order_delivered_customer_date - order_purchase_timestamp (filter WHERE order_status = 'delivered').
    - Reliable data range: January 2017 to August 2018.
    - Use customer_unique_id (not customer_id) for counting distinct customers.
"""

## LLM Call Function

In [ ]:
def ask_querymind(question: str) -> str:
    """
    Send a natural language question to Claude and get back SQL.
    Reminder: this is hard-coded - no RAG, no safety guardrails. 
    """
    message = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=1024,
        messages=[
            {
                "role": "user",
                "content": f"""You are an SQL expert. Given the database schema below, generate a single SQLite-compatible SELECT query to answer the user's question.
return ONLY the raw SQL query. No explanation, no markdown formatting, no code blocks, no backticks, just the SQL.

{SCHEMA_CONTEXT}

USER QUESTION: {question}

SQL:"""
            }
        ]
    )

    # Extract text from Claude's response
    sql = message.content[0].text.strip()
    return sql

## Experiments

In [5]:
# Test question 1 - Simple aggregation, single table
question = "How many orders were placed in 2017?"

print(f"Question: {question}\n")

# Step 1: Get SQL from Claude
sql = ask_querymind(question)
print(f"Generated SQL:\n{sql}\n")

# Step 2: Execute it
with engine.connect() as conn:
    try:
        result = pd.read_sql(sql, conn)
        print(f"Result:")
        display(result)
    except Exception as e:
        print(f"Execution error: {e}")

Question: How many orders were placed in 2017?

Generated SQL:
SELECT COUNT(*) FROM olist_orders WHERE order_purchase_timestamp LIKE '2017%'

Result:


,COUNT(*)
0,45101


In [6]:
# Test question 2 - Harder question, multi-table join
question = "What are the top 5 product categories by total revenue?"

print(f"Question: {question}\n")

sql = ask_querymind(question)
print(f"Generated SQL:\n{sql}\n")

with engine.connect() as conn:
    try:
        result = pd.read_sql(sql, conn)
        print(f"Result:")
        display(result)
    except Exception as e:
        print(f"Execution error: {e}")

Question: What are the top 5 product categories by total revenue?

Generated SQL:
SELECT pct.product_category_name_english, SUM(oi.price + oi.freight_value) as total_revenue
FROM olist_order_items oi
JOIN olist_products p ON oi.product_id = p.product_id
LEFT JOIN product_category_name_translation pct ON p.product_category_name = pct.product_category_name
GROUP BY pct.product_category_name_english
ORDER BY total_revenue DESC
LIMIT 5

Result:


,product_category_name_english,total_revenue
0,health_beauty,1441248.07
1,watches_gifts,1305541.61
2,bed_bath_table,1241681.72
3,sports_leisure,1156656.48
4,computers_accessories,1059272.40


In [7]:
# Test question 3 - test with a date + aggregation question

question = "What was the average delivery time in days for orders delivered in 2018?"

print(f"Question: {question}\n")

sql = ask_querymind(question)
print(f"Generated SQL:\n{sql}\n")

with engine.connect() as conn:
    try:
        result = pd.read_sql(sql, conn)
        print(f"Result:")
        display(result)
    except Exception as e:
        print(f"Execution error: {e}")

Question: What was the average delivery time in days for orders delivered in 2018?

Generated SQL:
SELECT AVG(JULIANDAY(order_delivered_customer_date) - JULIANDAY(order_purchase_timestamp)) as avg_delivery_days
FROM olist_orders 
WHERE order_status = 'delivered' 
AND order_delivered_customer_date IS NOT NULL 
AND order_purchase_timestamp LIKE '2018%'

Result:


,avg_delivery_days
0,12.137693


## Key Takeaways

Proved the basic loop works - Proof Of Concept: 

    Question --> LLM --> SQL --> Execute --> Results

**What this proves:**
* Claude can generate valid SQLitte-compatible SQL from schema context (simplified hard-coded).
* Schema conetxt contains enough information for the generation of accurate queries.
* Multi-table joins and date arithmetic work.

**Next steps**
* RAG retrieval insead of hardcoded schmea (Phase 2).
* SQL safety validation with sqlglot (Phase 3).
* Auto-visualization witth Plotly (Phase 4).
* Streamlit UI (Phase 4).